#### 251008: preparing the script to summarize the results of AXps generated for Boosted Rules

In [ ]:
import json
import os
import numpy as np
import re

# Step 1: summarize all raw log files information into the JSON format
# Information includes Encoding (#Vars, #Clauses), Runing time (max, min, avg)
# Two kind of json file => one contains all details for each instance, one is the summary
def extract_folder_info(folder, info_dict):
    r = re.compile('([a-zA-Z]+)([0-9]+)')
    folder_info = {}

    for info in folder.split('/')[-1].split('_')[1:]:
        info_k, info_v = r.match(info).groups()
        folder_info[info_dict[info_k]] = info_v
    return folder_info

def extract_summary_detail_json(results_bpath, info_dict):
    summary_json_path = os.path.join(results_bpath, 'summary.json')
    detail_json_path = os.path.join(results_bpath, 'detail.json')
    if os.path.exists(summary_json_path):
        os.remove(summary_json_path)
    if os.path.exists(detail_json_path):
        os.remove(detail_json_path)

    sub_folders = sorted([f.path for f in os.scandir(results_bpath) if f.is_dir()])
    res_summary_json, res_detail_json = [], []

    for sub_f in sub_folders:
        # get the corresponding infos in the name of sub_f
        sub_f_info_dict = extract_folder_info(sub_f, info_dict[0])

        files = sorted([f.path for f in os.scandir(sub_f) if f.is_file()])

        for file in files:
            curr_res_detail_json_i = []
            res_summary_dict = sub_f_info_dict.copy()
            res_summary_dict[info_dict[-1]] = file.split('/')[-1].split('.')[0]
            res_detail_dict = res_summary_dict.copy()

            with open(file, 'r') as f:
                lines = [l.strip('\n') for l in f.readlines() if l.strip()]
                for l in lines:
                    k, v = l.split(':')
                    # 251210: change for more information
                    infos = ['vars', 'clauses', 'max time', 'min time', 'avg time', 'avg calls', \
                             'avg vars', 'avg cos', 'avg cps']
                    if k in infos:
                        res_summary_dict[k] = v.strip()
                    else:
                        res_detail_dict[k] = v.strip()

                    if k == 'mem':
                        # avoid the instances duplicated 
                        if not res_detail_dict['i'] in curr_res_detail_json_i:
                            res_detail_json.append(res_detail_dict.copy())
                            curr_res_detail_json_i.append(res_detail_dict.copy()['i'])
            res_summary_dict['n_instance'] = len(curr_res_detail_json_i)
            res_summary_json.append(res_summary_dict)

    with open(summary_json_path, 'w') as f:
        json.dump(res_summary_json, f, indent=2)
    print("Summary Json for " + results_bpath + " is stored in " + summary_json_path)
    with open(detail_json_path, 'w') as f:
        json.dump(res_detail_json, f, indent=2)
    print("Deatiled for " + results_bpath + " is stored in " + detail_json_path)


In [ ]:
results_axp_boomer = 'results_AXp_boomer_251007'
results_axp_xgboost = 'results_AXp_xgboost_250925'
info_dict = [{'md': 'max_depth', 'num': 'n_estimator'}, 'dataset']

# call the function to generate the two json files
extract_summary_detail_json(results_axp_boomer, info_dict)
extract_summary_detail_json(results_axp_xgboost, info_dict)

#######################
import pandas as pd
import builtins
from collections import Counter, OrderedDict

# Step 2: Transform the results in JSON format into pandas format

df_xgboost = pd.read_json(os.path.join(results_axp_xgboost, 'summary.json'))
df_boomers = pd.read_json(os.path.join(results_axp_boomer, 'summary.json'))

df_xgboost['method'] = 'xgboost'
df_boomers['method'] = 'boomer'

df_xgboost_boomers = pd.concat([df_xgboost, df_boomers])
df_xgboost_boomers.info()
add_info_format = lambda r: r['dataset'] + '(' + str(r['n_instance']) + ')'
df_xgboost_boomers['dataset'] = df_xgboost_boomers.apply(add_info_format, axis='columns')

In [ ]:
# 251210: considering the results with different weight digits + pb_encoding + encoding
results_boomer_sat = 'results_AXp_boomer_sat'
results_boomer_mx = 'results_AXp_boomer_mx'
info_dict = [{'md': 'max_depth', 'num': 'n_estimator', 'wd': 'wd', 'pb': 'pb'}, 'dataset']
# call function to generate the json files
extract_summary_detail_json(results_boomer_sat, info_dict)
extract_summary_detail_json(results_boomer_mx, info_dict)

#######################
import pandas as pd
import builtins
from collections import Counter, OrderedDict

# Step 2: Transform the results in JSON format into pandas format
df_boomers_mx= pd.read_json(os.path.join(results_boomer_mx, 'summary.json'))
df_boomers_sat = pd.read_json(os.path.join(results_boomer_sat, 'summary.json'))

df_boomers_mx['method'] = 'mx'
df_boomers_sat['method'] = 'sat'

df_boomers = pd.concat([df_boomers_mx, df_boomers_sat])
df_boomers.info()
add_info_format = lambda r: r['dataset'] + '(' + str(r['n_instance']) + ')'
df_boomers['dataset'] = df_boomers.apply(add_info_format, axis='columns')

Summary Json for results_AXp_boomer_sat is stored in results_AXp_boomer_sat/summary.json
Deatiled for results_AXp_boomer_sat is stored in results_AXp_boomer_sat/detail.json
Summary Json for results_AXp_boomer_mx is stored in results_AXp_boomer_mx/summary.json
Deatiled for results_AXp_boomer_mx is stored in results_AXp_boomer_mx/detail.json
<class 'pandas.core.frame.DataFrame'>
Index: 2520 entries, 0 to 1259
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   max_depth    2520 non-null   int64  
 1   n_estimator  2520 non-null   int64  
 2   wd           2520 non-null   int64  
 3   pb           2520 non-null   int64  
 4   dataset      2520 non-null   object 
 5   avg vars     2520 non-null   float64
 6   avg cos      2520 non-null   float64
 7   max time     2518 non-null   float64
 8   min time     2518 non-null   float64
 9   avg time     2518 non-null   float64
 10  avg calls    2518 non-null   float64
 11  n_ins

#### Codes for preparing Tables!!

In [ ]:
## Functions preparing the Tables
def get_df_infos(df, method, max_depth, n_estimator, dataset=None, wd=None):
    '''
        updated in 260112 => add the wd, while wd must accompany with valid dataset
    '''
    if not dataset is None:
        assert (method in set(df['method'])) and (max_depth in set(df['max_depth'])) \
            and (n_estimator in set(df['n_estimator']) and dataset in set(df['dataset']))
        if not wd is None:
            assert (wd in set(df['wd']))
            df_infos = df[(df['method'] == method) & (df['max_depth'] == max_depth) \
                & (df['n_estimator'] == n_estimator) & (df['dataset'] == dataset) & df['wd'] == wd]
        else:
            df_infos = df[(df['method'] == method) & (df['max_depth'] == max_depth) \
                & (df['n_estimator'] == n_estimator) & (df['dataset'] == dataset)]
        assert df_infos.shape[0] == 1
    else:
        assert (method in set(df['method'])) and (max_depth in set(df['max_depth'])) \
            and (n_estimator in set(df['n_estimator']))
        df_infos = df[(df['method'] == method) & (df['max_depth'] == max_depth) \
                & (df['n_estimator'] == n_estimator)]
    return df_infos

def str_with_bold_value(df1, df2, col_names, maxs):
    def value2str(v):
        # considering the 0.00 cases (< 0.01) => especially in seconds
        if isinstance(v, np.float64):
            return f'{v:.2f}' if v > 0.01 else '$<$'
        elif isinstance(v, np.int64):
            return str(v)
        else:
            return v
        
    assert len(col_names) == len(maxs)
    better_infos = []
    str_row = ''
    # 251002: add the value #wins (if equal, both are #wins + 1)
    wins_d1, wins_d2 = {}, {}
    for i in range(len(col_names)):
        wins_d1[col_names[i]] = 0
        wins_d2[col_names[i]] = 0
        if df1.iloc[-1][col_names[i]] == df2.iloc[-1][col_names[i]]:
            better_infos.append(False)
            str_row += '\\textbf{' + value2str(df1.iloc[-1][col_names[i]]) + '} & ' 
            wins_d1[col_names[i]] += 1
            wins_d2[col_names[i]] += 1
        elif (df1.iloc[-1][col_names[i]] > df2.iloc[-1][col_names[i]]) ^ maxs[i]:
            better_infos.append(False)
            str_row += value2str(df1.iloc[-1][col_names[i]]) + ' & '
            wins_d2[col_names[i]] += 1
        else:
            better_infos.append(True)
            str_row += '\\textbf{' + value2str(df1.iloc[-1][col_names[i]]) + '} & '
            wins_d1[col_names[i]] += 1
    for i in range(len(col_names) - 1):
        str_row += value2str(df2.iloc[-1][col_names[i]]) + ' & ' if better_infos[i] else '\\textbf{' + value2str(df2.iloc[-1][col_names[i]]) + '} & '
    str_row += value2str(df2.iloc[-1][col_names[-1]]) + ' \\\\' if better_infos[-1] else '\\textbf{' + value2str(df2.iloc[-1][col_names[-1]]) + '} \\\\'
    return str_row, wins_d1, wins_d2

def deal_wins(win_all, win_specific):
    if win_all is None:
        win_all = Counter(win_specific)
    else:
        win_all = win_all + Counter(win_specific)
    return win_all

def wins2str(wins, keys_given):
    wins2str = ''
    for i in range(len(keys_given) - 1):
        wins2str += str(wins[keys_given[i]]) + ' & '
    wins2str += str(wins[keys_given[-1]]) + ' '
    return wins2str

In [ ]:
## 2511: Code preparing tables comparing Boomers and XGBoost
def prepare_table_by_bench_separ(df, out_base_path='latex_separ'):
    '''
        Not changed!!
    '''
    if not os.path.exists(out_base_path):
        os.mkdir(out_base_path)
    datasets_infos = sorted(set(df['dataset']))
    cols_selected = ['train_acc', 'test_acc', 'num_identical_path']

    for d_info in datasets_infos:
        out_path = os.path.join(out_base_path, d_info.replace('\n', '') + '_separ.tex')
        win_b_all, win_x_all = None, None
        # 1st col (m_depth) => 2nd col (num_estimator) => Boomer(train/test/num_identical_path), XGBoost
        num_estimators = sorted(set(df['n_estimator']))
        with builtins.open(out_path, 'w') as f:
            for m_depth in sorted(set(df['max_depth'])):
                # prepare each multirow cell
                f.write('\\multirow{' + str(len(num_estimators)) + '}{*}{' + str(m_depth) + '}\n')
                for n in num_estimators:
                    f.write('& ' + str(n) + ' & ')
                    df_infos_boomer = get_df_infos(df, 'boomer', m_depth, n, d_info)
                    df_infos_xgboost = get_df_infos(df, 'xgboost', m_depth, n, d_info)
                    str_row, win_boomer, win_xgboost = str_with_bold_value(df_infos_boomer, df_infos_xgboost, cols_selected, [True, True, False])
                    f.write(str_row + '\n')
                    win_b_all = deal_wins(win_b_all, win_boomer)
                    win_x_all = deal_wins(win_x_all, win_xgboost)
                f.write('\\hline \n')
            # dealing the last line of #WINS
            f.write('\\#W & / & ' + wins2str(win_b_all, cols_selected) + ' & ' + wins2str(win_x_all, cols_selected) + '\\\\ \\hline')

def prepare_table_by_bench_specific(df, out_bpath, col_selected_dict, num_estimator=50):
    '''
        col_selected_dict should be ordered
    '''
    if not os.path.exists(out_bpath):
        os.mkdir(out_bpath)
    out_path = os.path.join(out_bpath, '0_results_boomer_xgboost_base(' + str(num_estimator) + ').tex')
    datasets = sorted(set(df['dataset']))
    mdepths = sorted(set(df['max_depth']))
    win_b_all, win_x_all = None, None

    with builtins.open(out_path, 'w') as f:
        # 1st col (datasets) => 2nd col (max_depths) => Boomer (train/test/num_identical_path), XGBoost
        for d_info in datasets:
            f.write('\\multirow{' + str(len(mdepths)) + '}{*}{\\shortstack[1]{' + d_info + '}}\n')
            for m_dpeth in mdepths:
                f.write('& ' + str(m_dpeth) + ' & ')
                df_infos_boomer = get_df_infos(df, 'boomer', m_dpeth, num_estimator, d_info)
                df_infos_xgboost = get_df_infos(df, 'xgboost', m_dpeth, num_estimator, d_info)
                str_row, win_boomer, win_xgboost = str_with_bold_value(df_infos_boomer, df_infos_xgboost, list(col_selected_dict.keys()), list(col_selected_dict.values()))
                win_b_all = deal_wins(win_b_all, win_boomer)
                win_x_all = deal_wins(win_x_all, win_xgboost)
                f.write(str_row + '\n')
            f.write('\\hline \n')
        # dealing the last line of #WINS
        f.write('\\#WINS & / & ' + wins2str(win_b_all, list(col_selected_dict.keys())) + ' & ' + wins2str(win_x_all, list(col_selected_dict.keys())) + '\\\\ \\hline')

# Step 3: preparing the table of given num_estimator
col_selected_dict = OrderedDict({'vars': False, 'clauses': False, 'max time': False, 'min time': False, 'avg time': False, 'avg calls': False})
nums = sorted(set(df_xgboost_boomers['n_estimator']))
for n in nums:
    prepare_table_by_bench_specific(df_xgboost_boomers, 'tex_table', col_selected_dict, num_estimator=n)


In [ ]:
## 260112: Changed code to prepare tables comparing Boomers with different encodings! (SAT/MaxSAT)
def prepare_table_by_bench_separ(df, out_base_path='latex_separ'):
    '''
        Table for comparing maxsat & sat encoding for Boomer => for a specific benchmark
    '''
    if not os.path.exists(out_base_path):
        os.mkdir(out_base_path)
    datasets_infos = sorted(set(df['dataset']))
    cols_selected = ['', 'test_acc', 'num_identical_path']

    for d_info in datasets_infos:
        out_path = os.path.join(out_base_path, d_info.replace('\n', '') + '_separ.tex')
        win_b_all, win_x_all = None, None
        # 1st col (m_depth) => 2nd col (num_estimator) => Boomer(train/test/num_identical_path), XGBoost
        num_estimators = sorted(set(df['n_estimator']))
        with builtins.open(out_path, 'w') as f:
            for m_depth in sorted(set(df['max_depth'])):
                # prepare each multirow cell
                f.write('\\multirow{' + str(len(num_estimators)) + '}{*}{' + str(m_depth) + '}\n')
                for n in num_estimators:
                    f.write('& ' + str(n) + ' & ')
                    df_infos_boomer = get_df_infos(df, 'boomer', m_depth, n, d_info)
                    df_infos_xgboost = get_df_infos(df, 'xgboost', m_depth, n, d_info)
                    str_row, win_boomer, win_xgboost = str_with_bold_value(df_infos_boomer, df_infos_xgboost, cols_selected, [True, True, False])
                    f.write(str_row + '\n')
                    win_b_all = deal_wins(win_b_all, win_boomer)
                    win_x_all = deal_wins(win_x_all, win_xgboost)
                f.write('\\hline \n')
            # dealing the last line of #WINS
            f.write('\\#W & / & ' + wins2str(win_b_all, cols_selected) + ' & ' + wins2str(win_x_all, cols_selected) + '\\\\ \\hline')

def prepare_table_by_bench_specific(df, out_bpath, col_selected_dict, num_estimator=50):
    '''
        col_selected_dict should be ordered
    '''
    if not os.path.exists(out_bpath):
        os.mkdir(out_bpath)
    out_path = os.path.join(out_bpath, '0_results_boomer_xgboost_base(' + str(num_estimator) + ').tex')
    datasets = sorted(set(df['dataset']))
    mdepths = sorted(set(df['max_depth']))
    win_b_all, win_x_all = None, None

    with builtins.open(out_path, 'w') as f:
        # 1st col (datasets) => 2nd col (max_depths) => Boomer (train/test/num_identical_path), XGBoost
        for d_info in datasets:
            f.write('\\multirow{' + str(len(mdepths)) + '}{*}{\\shortstack[1]{' + d_info + '}}\n')
            for m_dpeth in mdepths:
                f.write('& ' + str(m_dpeth) + ' & ')
                df_infos_boomer = get_df_infos(df, 'boomer', m_dpeth, num_estimator, d_info)
                df_infos_xgboost = get_df_infos(df, 'xgboost', m_dpeth, num_estimator, d_info)
                str_row, win_boomer, win_xgboost = str_with_bold_value(df_infos_boomer, df_infos_xgboost, list(col_selected_dict.keys()), list(col_selected_dict.values()))
                win_b_all = deal_wins(win_b_all, win_boomer)
                win_x_all = deal_wins(win_x_all, win_xgboost)
                f.write(str_row + '\n')
            f.write('\\hline \n')
        # dealing the last line of #WINS
        f.write('\\#WINS & / & ' + wins2str(win_b_all, list(col_selected_dict.keys())) + ' & ' + wins2str(win_x_all, list(col_selected_dict.keys())) + '\\\\ \\hline')


#### The code for Plotting!!

In [ ]:
import matplotlib.pyplot as plt 
from itertools import accumulate

df_xgboost_d = pd.read_json(os.path.join(results_axp_xgboost, 'detail.json'))
df_boomers_d = pd.read_json(os.path.join(results_axp_boomer, 'detail.json'))

df_xgboost_d['method'] = 'xgboost'
df_boomers_d['method'] = 'boomer'

df_xgboost_boomers_d = pd.concat([df_xgboost_d, df_boomers_d])

# Step 4: Plotting figure of raw performance (instances explianed - CPU time) using Json file of detailed information
def plot_linechart_given_dataframe(df, max_depth=5, n_estimator=40, s=False):
    df_x_extracted = get_df_infos(df, 'xgboost', max_depth, n_estimator)
    df_b_extracted = get_df_infos(df, 'boomer', max_depth, n_estimator)
    assert df_x_extracted.shape[0] == df_b_extracted.shape[0]
    num_instance = df_x_extracted.shape[0]

    if not s:
        time_x = sorted(df_x_extracted['t'])
        time_b = sorted(df_b_extracted['t'])
    else:
        time_x = list(accumulate(sorted(df_x_extracted['t'])))
        time_b = list(accumulate(sorted(df_b_extracted['t'])))
    
    plt.figure(figsize=(20, 6))
    plt.xlim(3200, 3810)
    plt.plot(list(range(1, num_instance+1)), time_x, marker='o', label='MaxSAT-XGBoost', markersize=5)
    plt.plot(list(range(1, num_instance+1)), time_b, marker='x', label='MaxSAT-Boomer', markersize=5)

    plt.xlabel('Number of Instances Explained')
    plt.ylabel('CPU Time (seconds)')
    plt.legend()
    plt.grid(visible=True)
    if s:
        t = "Raw performance of *sum time* for max_depth=" + str(max_depth) + " and num=" + str(n_estimator)
        fig_path = './figures/' + 'raw_md' + str(max_depth) + '_num' + str(n_estimator) + '_sum.png'
    else:
        t = "Raw performance of *separate time* for max_depth=" + str(max_depth) + " and num=" + str(n_estimator)
        fig_path = './figures/' + 'raw_md' + str(max_depth) + '_num' + str(n_estimator) + '_separate.png'
    plt.title(t)
    plt.savefig(fig_path)
    plt.show()


In [ ]:
# Plotting the image of different (max_depth * n_estimator) cases separately
m_depths = sorted(set(df_xgboost_boomers_d['max_depth']))

for n in nums:
    for m_depth in m_depths:
        # print("n={}, m_depth={}".format(n, m_depth))
        plot_linechart_given_dataframe(df_xgboost_boomers_d, max_depth=m_depth, n_estimator=n)
        plot_linechart_given_dataframe(df_xgboost_boomers_d, max_depth=m_depth, n_estimator=n, s=True)
        

In [ ]:
# Step 4.2: plot the linechart of multiple (max_depth * n_estimator) cases
def plot_multi_linechart_given_dataframe(df, save_path, title='Multiple Cases', info_dict={'xgboost': [(5, 50)], 'boomer': [(5, 50)]}, markers={'xgboost': 'o', 'boomer': 'x'}, s=False):
    
    plt.figure(figsize=(20, 8))
    plt.xlim(3200, 3810)
    plt.xlabel('Number of Instances Explained')
    plt.ylabel('CPU Time (seconds)')
    plt.grid(visible=True)

    for m, value_list in info_dict.items():
        for m_depth, n in value_list:
            df_extracted = get_df_infos(df, m, m_depth, n)
            num_instance = df_extracted.shape[0]

            if not s:
                time = sorted(df_extracted['t'])
            else:
                time = list(accumulate(sorted(df_extracted['t'])))
            label = 'MaxSAT-' + m + '_md' + str(m_depth) + '_n' + str(n)
            plt.plot(list(range(1, num_instance+1)), time, marker=markers[m], label=label, markersize=4)

    plt.title(title)
    plt.legend()
    plt.savefig(save_path)
    plt.show()

In [ ]:
# Considering multiple cases of separated n_estimators
info_method_list = lambda m_depths, n: [(m, n) for m in m_depths]

for n in nums:
    info_dict = {'xgboost': info_method_list(m_depths, n), 'boomer': [(m_depths[-1], n)]}
    title = 'Raw Performance of *separate time* for all max_depths of num=' + str(n)
    fig_path = './figures/raw_multiple_num' + str(n) + '_separate.png'
    plot_multi_linechart_given_dataframe(df_xgboost_boomers_d, fig_path, title=title, info_dict=info_dict)
    title = 'Raw Performance of *sum time* for all max_depths of num=' + str(n)
    fig_path = './figures/raw_multiple_num' + str(n) + '_sum.png'
    plot_multi_linechart_given_dataframe(df_xgboost_boomers_d, fig_path, title=title, info_dict=info_dict, s=True)


In [ ]:
# Considering multiple cases of separated n_estimators
info_method_list_md = lambda md, nums: [(md, n) for n in nums]

for md in m_depths:
    info_dict = {'xgboost': info_method_list_md(md, nums), 'boomer': [(md, nums[-1])]}
    title = 'Raw Performance of *separate time* for all n_estimators of max_depth=' + str(md)
    fig_path = './figures/raw_multiple_md' + str(md) + '_separate.png'
    plot_multi_linechart_given_dataframe(df_xgboost_boomers_d, fig_path, title=title, info_dict=info_dict)
    title = 'Raw Performance of *sum time* for all max_depths of max_depth=' + str(md)
    fig_path = './figures/raw_multiple_md' + str(md) + '_sum.png'
    plot_multi_linechart_given_dataframe(df_xgboost_boomers_d, fig_path, title=title, info_dict=info_dict, s=True)
